# Objectives 

- Interacting atomically with two Dapps



# Interacting atomically with two Dapps


## Hotel and Flight contracts

Assume you are going on a vacation and need to book travel and accommodation. We have created two contracts mimicking a decentralized application for booking accommodation and travel. The hotel reservation contract can be found [here](https://sepolia.etherscan.io/address/0x3e3034173f8227aaa996ae8d22b2ff2fed9653d8). Let us go through its main ``bookRoom()`` function

- Initial few checks ensure the payment is sufficient and rooms are available at the hotel
- The state of the contract is updated to make sure the room is booked in the name of the wallet initiating the transaction calling the function
- The last part calculates and disburses a refund if the transaction caller sent more money than the ticket price of `10**10 wei`.
- Note that no part of the contract allows for cancellation of the reservation; this Dapp corresponds to a Non-refundable reservation

You will now book accommodation for the trip. 
- Go to the contract on Etherscan, click the write contract tab, and connect your wallet. - - Click the ``bookRoom`` tab. 
- Set the amount of Wei to send as a payment. The required payment is `10000000000 wei` or `0.00000001 ETH` (recommended); you can send more if you want, and the contract will refund it within the same transaction.
- Check the status of your reservation by querying `CheckReservations` with your public key

Now that we have booked the accommodation, let us book a flight. The flight reservation contract looks similar to the hotel reservation contract with ``bookFlight()`` replacing ``bookRoom()``. You can access the flight reservation contract [here](https://sepolia.etherscan.io/address/0x0278755e8f1508631Fc28eDe9Bf17e5cb0336741). Perform the same steps as the hotel reservation contract to book a flight ticket. Check the status of the transaction on Etherscan.

Notice that the transaciton will fail for some of you since there aren't enough flight tickets available for all class members. This leads to a situation where your hotel reservation is practically useless since there is no way to travel. Is it possible to avoid such a situation: *Book hotel only if the flight is available* and *Book flight only if the hotel is available*? In other words, we want the trip reservation to be atomic - Either book both travel and accommodation or book none. Notice that the Dapps are independent of one another - think of them as two separate travel websites. 

## Atomic transaction

We can perform such transactions easily on blockchains. Note that an ethereum transaction either succeeds or fails and reverts back. Thus, we need to combine ``bookRoom()`` and ``bookFlight()`` into a single transaction. 

We can do this by launching a smart contract whose one function call will call two of the above hotel and flight contracts. The atomic transaction contract (aka trip reservation contract) can be found [here](https://sepolia.etherscan.io/address/0x380a77981b53546dce54502e308a02f11c9252af). Let us quickly go through the ``bookTrip`` function:
- The function takes in 4 arguments - the addresses of Hotel and Flight reservation contracts and their prices
- It then calls both the hotel and flight contract 
- The calls are sequential, however, due to the atomic nature of an Ethereum transaction, the whole ``bookTrip`` transaction will fail if either of the contract calls fails.
- The last part disburses any remaining refund.

I have refreshed both the hotel and flight contracts, and you will now use the atomic transaction contract to book both atomically:
- Go to the contract on Etherscan, click the write contract tab, and connect your wallet. - - Click the ``bookTrip`` tab. 
- Set the amount of ETH to send as a payment. The required total payment is `20000000000 wei` or `0.00000002 ETH`; anything extra will be refunded. 
- Give the contract addresses of the hotel and flight reservation contract; set the price of both to ``10000000000`` wei.
- Check the status of the transaction on Etherscan

The transaction will succeed for some of you and fail for the rest. Check the status of your booking on the respective contracts. If the atomic transaction failed, you should not have confirmed reservations on both the hotel and flight contracts. 

In [5]:
# Lab 3: Interacting atomically with two Dapps (Sepolia testnet)
#
# This cell performs the three on-chain steps of the lab:
#   1. bookRoom()   on the HotelReservation  contract
#   2. bookFlight() on the FlightReservation contract
#   3. bookTrip()   on the AtomicTripReservation contract (atomic: both or neither)
#
# NOTE: every run of this cell sends NEW transactions. Reservations are keyed
# per saleIteration, so re-booking in the same iteration is refunded on-chain.

import os
from datetime import datetime, timedelta
from web3 import Web3

w3 = Web3(Web3.HTTPProvider('https://sepolia.infura.io/v3/202379f0e9fe40e6aeac54fbfb0b6a49'))
assert w3.is_connected(), 'not connected to Sepolia'
print('connected, block:', w3.eth.block_number)

# Sepolia's suggested gas price can sit below the moving base fee, leaving a
# transaction stuck in the mempool. 2x with a 3 gwei floor mines reliably.
GAS_PRICE = max(2 * w3.eth.gas_price, Web3.to_wei(3, 'gwei'))
print('gas price:', Web3.from_wei(GAS_PRICE, 'gwei'), 'gwei')

# Wallet details
public_address = '0xf10d396C334FAD007761db6D134Ccd09cee20C48'  # your MetaMask address
# The private key is read from the environment so it never lands in this
# (git-tracked) notebook: export LAB3_PRIVATE_KEY=0x... before running.
private_key = os.environ.get('LAB3_PRIVATE_KEY', '')
assert private_key.startswith('0x'), 'set the LAB3_PRIVATE_KEY environment variable'

# Contract addresses from the lab README
HOTEL  = Web3.to_checksum_address('0x3e3034173f8227aaa996ae8d22b2ff2fed9653d8')
FLIGHT = Web3.to_checksum_address('0x0278755e8f1508631Fc28eDe9Bf17e5cb0336741')
TRIP   = Web3.to_checksum_address('0x380a77981b53546dce54502e308a02f11c9252af')
PRICE  = 10**10  # 0.00000001 ETH, ticket price of each Dapp

# Fetch the nonce ONCE and increment it locally for each send. Re-querying
# between transactions can race Infura's load-balanced nodes (a node that
# has not processed the previous block yet returns a stale nonce, and the
# next send fails with 'nonce too low').
nonce = w3.eth.get_transaction_count(public_address, 'pending')

def send(contract_call, value, label):
    global nonce
    txn = contract_call.build_transaction({
        'nonce': nonce,
        'from': public_address,
        'value': value,
        'gasPrice': GAS_PRICE,
        'chainId': 11155111,  # Sepolia
    })
    txn['gas'] = w3.eth.estimate_gas(txn)
    signed_txn = w3.eth.account.sign_transaction(txn, private_key)
    txn_hash = w3.eth.send_raw_transaction(signed_txn.raw_transaction)
    print(label, 'txn hash:', Web3.to_hex(txn_hash))
    receipt = w3.eth.wait_for_transaction_receipt(txn_hash, timeout=300)
    print(label, 'status:', 'SUCCESS' if receipt.status == 1 else 'FAILED',
          '| block:', receipt.blockNumber, '| gasUsed:', receipt.gasUsed)
    nonce += 1
    return receipt

# --- Step 1: book a hotel room directly ---------------------------------
hotel_abi = '[{"inputs":[],"stateMutability":"nonpayable","type":"constructor"},{"inputs":[{"internalType":"uint256","name":"numRooms","type":"uint256"}],"name":"blockRooms","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"bookRoom","outputs":[],"stateMutability":"payable","type":"function"},{"inputs":[{"internalType":"address","name":"CustomerAddress","type":"address"}],"name":"checkReservations","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"manager","outputs":[{"internalType":"address payable","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"numReservations","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint256","name":"","type":"uint256"},{"internalType":"address","name":"","type":"address"}],"name":"reservations","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint256","name":"newPrice","type":"uint256"},{"internalType":"uint256","name":"newAvailability","type":"uint256"}],"name":"resetState","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"rooms","outputs":[{"internalType":"uint256","name":"availability","type":"uint256"},{"internalType":"uint256","name":"price","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"saleIteration","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"transferEarningsToManager","outputs":[],"stateMutability":"nonpayable","type":"function"}]'
hotel = w3.eth.contract(HOTEL, abi=hotel_abi)

print('rooms available:', hotel.functions.rooms().call())
send(hotel.functions.bookRoom(), PRICE, 'bookRoom')

# --- Step 2: book a flight directly --------------------------------------
flight_abi = '[{"inputs":[],"stateMutability":"nonpayable","type":"constructor"},{"inputs":[{"internalType":"uint256","name":"numSeats","type":"uint256"}],"name":"blockSeats","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"bookFlight","outputs":[],"stateMutability":"payable","type":"function"},{"inputs":[{"internalType":"address","name":"CustomerAddress","type":"address"}],"name":"checkReservations","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"flights","outputs":[{"internalType":"uint256","name":"availability","type":"uint256"},{"internalType":"uint256","name":"price","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"manager","outputs":[{"internalType":"address payable","name":"","type":"address"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"numReservations","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint256","name":"","type":"uint256"},{"internalType":"address","name":"","type":"address"}],"name":"reservations","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"view","type":"function"},{"inputs":[{"internalType":"uint256","name":"newPrice","type":"uint256"},{"internalType":"uint256","name":"newAvailability","type":"uint256"}],"name":"resetState","outputs":[],"stateMutability":"nonpayable","type":"function"},{"inputs":[],"name":"saleIteration","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"},{"inputs":[],"name":"transferEarningsToManager","outputs":[],"stateMutability":"nonpayable","type":"function"}]'
flight = w3.eth.contract(FLIGHT, abi=flight_abi)

print('seats available:', flight.functions.flights().call())
send(flight.functions.bookFlight(), PRICE, 'bookFlight')

# --- Step 3: book the trip atomically -------------------------------------
trip_abi = '[{"inputs":[{"internalType":"address","name":"HotelReservationAddress","type":"address"},{"internalType":"address","name":"FlightReservationAddress","type":"address"},{"internalType":"uint256","name":"HotelPrice","type":"uint256"},{"internalType":"uint256","name":"FlightPrice","type":"uint256"}],"name":"bookTrip","outputs":[],"stateMutability":"payable","type":"function"}]'
trip = w3.eth.contract(TRIP, abi=trip_abi)

# Because the reservation Dapps are independent of each other, two separate
# transactions can leave us with a hotel room but no flight (rooms are
# non-refundable). bookTrip() wraps both calls in ONE transaction: if either
# internal call fails, the whole transaction reverts and nothing is booked.
send(trip.functions.bookTrip(HOTEL, FLIGHT, PRICE, PRICE), 2 * PRICE, 'bookTrip')

# --- Verify reservations ---------------------------------------------------
print()
print('=== Reservation status for', public_address, '===')
print('hotel booked: ', hotel.functions.checkReservations(public_address).call())
print('flight booked:', flight.functions.checkReservations(public_address).call())

# --- Results from the first execution of this lab (2026-09-15) -------------
# wallet: 0xf10d396C334FAD007761db6D134Ccd09cee20C48
# 1. bookRoom   -> 0x01cd053ca5199183a382bea2072549ba7003c32e24d838374efd6383b6668ef1  SUCCESS (block 11711399)
# 2. bookFlight -> 0x5a6f833438ff190b11cf1f3593656ad2303e20ce4d85e808044fead3d00cfa29  SUCCESS (block 11711400)
# 3. bookTrip   -> 0xa3e2e1e97471b87efe9d8e14c2ea3d9c797111dce4e1949e37e44724bc4e1806  SUCCESS (block 11711430)
# Final state: hotel booked = True, flight booked = True

connected, block: 11711516
gas price: 3 gwei


AssertionError: set the LAB3_PRIVATE_KEY environment variable